In [2]:
import sys
from pathlib import Path
import pandas as pd

# Set the project root manually
project_root = Path("C:/Users/dinat/Υπολογιστής/Pfizer Masterclass/airbnb-price-prediction-app")
sys.path.append(str(project_root))

In [3]:
data = pd.read_csv(project_root / 'data/listings.csv')

In [4]:
from backend.constants import cols_to_drop

data = data.drop(columns=cols_to_drop)

In [5]:
print(data.columns)

Index(['host_listings_count', 'host_total_listings_count',
       'host_verifications', 'neighbourhood_cleansed',
       'neighbourhood_group_cleansed', 'latitude', 'longitude',
       'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'calendar_updated',
       'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'instant_bookable'],
      dtype='object')


In [6]:
data[['availability_60', 'availability_90', 'availability_365']]

,availability_60,availability_90,availability_365
0,49,79,170
1,56,86,361
2,26,56,331
3,52,82,357
4,27,57,208
...,...,...,...
9577,51,81,341
9578,60,90,365
9579,54,84,359
9580,54,84,359


In [ ]:
## Geographic Analysis: Occupancy Rate by Neighbourhood

Occupancy rate is estimated as `(365 - availability_365) / 365 * 100`.  
Each bubble is centred on the neighbourhood's mean lat/lon; size reflects the number of listings, colour reflects average occupancy rate.


-237

In [ ]:
import plotly.express as px

# Calculate per-listing occupancy rate (same formula as the backend)
geo = data[['neighbourhood_cleansed', 'latitude', 'longitude', 'availability_365']].copy()
geo['availability_365'] = pd.to_numeric(geo['availability_365'], errors='coerce').clip(0, 365)
geo['occupancy_rate'] = (365 - geo['availability_365']) / 365 * 100

# Aggregate to neighbourhood level
nb_geo = (
    geo.groupby('neighbourhood_cleansed')
    .agg(
        lat=('latitude', 'mean'),
        lon=('longitude', 'mean'),
        avg_occupancy_rate=('occupancy_rate', 'mean'),
        listing_count=('occupancy_rate', 'count'),
    )
    .reset_index()
)
nb_geo['avg_occupancy_rate'] = nb_geo['avg_occupancy_rate'].round(1)

fig = px.scatter_mapbox(
    nb_geo,
    lat='lat',
    lon='lon',
    color='avg_occupancy_rate',
    size='listing_count',
    hover_name='neighbourhood_cleansed',
    hover_data={'avg_occupancy_rate': True, 'listing_count': True, 'lat': False, 'lon': False},
    color_continuous_scale='RdYlGn',
    size_max=40,
    zoom=11,
    mapbox_style='open-street-map',
    title='Average Occupancy Rate by Neighbourhood (bubble size = listing count)',
    labels={'avg_occupancy_rate': 'Avg Occupancy %', 'listing_count': 'Listings'},
    height=650,
)
fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig.show()
